# Atlas Integration: Cortical + Subcortical → Combined Parcellation

This notebook is a step-by-step explanation of how to merge two integer-labelled NIfTI atlases — one cortical, one subcortical — into a single combined atlas. The process involves loading the atlases, optionally resampling the subcortical atlas to match the cortical grid, calculating label offsets to avoid collisions, merging the label arrays according to a precedence rule, and saving the resulting combined atlas along with an optional CSV label-mapping table. The notebook also handles edge cases such as shape/affine mismatches and missing resampling backends, ensuring robustness throughout the integration process. Optional dependencies for resampling are noted, and debug logging is included to provide insights into the merging process. 

This integrated atlas is used throughout (almost) all subsequent analyses, including the merging and aggregation of rfMRI timeseries in `04_merge_aggregate_rfMRI.ipynb`, the aggregation of dMRI measures in `05_aggregate_dMRI.ipynb`, and the global and module-level clustering analyses in `06_global_clustering.ipynb` and `09_module_clustering.ipynb`, respectively.


## Purpose

Merge two integer-labelled NIfTI atlases — one cortical, one subcortical — into
a single combined atlas. The result is:

- A **combined NIfTI file** (int32, borrowing the cortical affine/header).
- An optional **CSV label-mapping table** recording the origin of every label
  in the combined atlas.

---

## Pipeline Steps

| # | Step | Key function |
|---|------|-------------|
| 1 | Load both NIfTI atlases; round float32 → int64 to fix resampling artefacts | `load_nifti()` |
| 2 | *(Optional)* Resample subcortical atlas to the cortical grid (nearest-neighbour) | `integrate_atlases(resample=True)` |
| 3 | Compute per-atlas unique non-zero labels; calculate an integer offset so labels never collide | `integrate_atlases()` internal |
| 4 | Remap each atlas's labels into the collision-free space; merge arrays using precedence rule | `integrate_atlases()` internal |
| 5 | *(Optional)* Load human-readable label names from JSON or delimited text files | `_load_label_names()` |
| 6 | Save combined array as int32 NIfTI and write CSV label table | `save_nifti()` / `integrate_atlases()` |

---

## Inputs

| Item | Description |
|------|-------------|
| **Cortical NIfTI** | Integer-labelled 3-D image in a common reference space (e.g. MNI152 1 mm). |
| **Subcortical NIfTI** | Integer-labelled 3-D image. Must share shape & affine with the cortical image *unless* `resample=True`. |
| **Cortical label file** *(optional)* | JSON `{"label": "name"}` or delimited text file (TSV/CSV/TXT) mapping integer labels to region names. |
| **Subcortical label file** *(optional)* | Same format as above for the subcortical atlas. |
| **Column selector** *(optional)* | 1-based column selection string for text label files, e.g. `'2'`, `'2,3'`, `'2-4'`. |
| **`cortex_precedence`** | Bool. If `True` (default), cortical labels win in overlapping voxels; subcortical labels are offset by `max(cortical labels)`. |
| **`resample`** | Bool. If `True`, the subcortical atlas is resampled to the cortical grid before merging. Requires `nibabel >= 3` or `nilearn`. |

---

## Outputs

| Item | Description |
|------|-------------|
| **Combined NIfTI** | int32, same shape/affine/header as the cortical atlas. |
| **CSV label table** | Columns: `new_label`, `source` (`'cortical'`/`'subcortical'`), `original_label`, `original_name`. |
| **Return value** | `dict` mapping `new_label → (source, original_label, original_name)`. |

---

## Edge Cases Handled

- **Shape/affine mismatch** → raises `ValueError` unless `resample=True`.
- **Zero voxels** treated as background throughout; never remapped.
- **Label collisions** avoided by offsetting the non-precedent atlas by the max label of the precedent atlas.
- **Missing resampling backend** → raises `RuntimeError` with an install hint.
- **Debug mode** → logs per-atlas label counts, voxel counts, and sample mapping entries via `logging`.

---

## Optional Dependencies

- `nibabel >= 3` — `nibabel.processing.resample_from_to` (preferred resampling backend).
- `nilearn` — `nilearn.image.resample_to_img` (fallback). If neither is installed, resampling is unavailable but the script runs normally when the two atlases already share a grid.

## 1 — Import Libraries and Setup Environment

The script depends on:

- **`nibabel`** — load and save NIfTI files.
- **`numpy`** — array manipulation (integer remapping, masking, statistics).
- **`nibabel.processing.resample_from_to`** *(optional)* — nearest-neighbour resampling; preferred backend.
- **`nilearn.image.resample_to_img`** *(optional)* — fallback resampling backend.
- **`argparse`, `csv`, `json`, `logging`, `os`** — standard-library utilities used by the CLI and helper functions.

The `try/except` blocks for the resampling imports mean the script degrades gracefully: if neither `nibabel >= 3` nor `nilearn` is available, resampling is simply unavailable (a `RuntimeError` is raised at runtime only if `resample=True` is actually requested).

In [ ]:
from __future__ import annotations

import csv
import json
import logging
import os
from typing import Dict, Optional, Tuple

import nibabel as nib
import numpy as np

# Resampling: prefer nibabel.processing.resample_from_to; fall back to nilearn if available.
try:
    from nibabel.processing import resample_from_to as _nib_resample
    print("Resampling backend: nibabel.processing.resample_from_to")
except Exception:
    _nib_resample = None
    print("nibabel resampling backend not available.")

try:
    from nilearn.image import resample_to_img as _nilearn_resample
    print("Resampling fallback: nilearn.image.resample_to_img")
except Exception:
    _nilearn_resample = None
    print("nilearn resampling backend not available.")

if _nib_resample is None and _nilearn_resample is None:
    print("WARNING: No resampling backend available. resample=True will raise RuntimeError at runtime.")

# Configure logging so INFO-level messages appear inline in the notebook.
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s", force=True)
logger = logging.getLogger(__name__)
print("All imports successful.")

## 2 — Configuration and Input Parameters

All paths and processing options are collected here in one place so the rest
of the notebook can run without modification.

### Atlas files used in this example

| Role | File |
|------|------|
| **Cortical atlas (NIfTI)** | Schaefer 2018 — 1000 parcels, 7 Networks, MNI152 1 mm |
| **Subcortical atlas (NIfTI)** | Tian 2020 Scale-IV (S4), 3 T, 1 mm |
| **Cortical label file** | Plain-text order file shipped with Schaefer parcellation |
| **Subcortical label file** | Plain-text label file shipped with Tian parcellation |

> **Grid note**: The Schaefer 1 mm and Tian S4 1 mm images should already share
> the MNI152 1 mm grid. Set `RESAMPLE = True` if you use atlases with different
> voxel sizes or FOVs.

### Key parameters

| Parameter | Default | Meaning |
|-----------|---------|---------|
| `CORTEX_PRECEDENCE` | `True` | Cortical labels win in overlapping voxels; subcortical labels are offset by `max(cortical labels)`. |
| `RESAMPLE` | `False` | Resample subcortical to cortical grid if shapes/affines differ. |
| `CORTEX_LABEL_COLUMNS` | `'2'` | Use column 2 from the Schaefer order file (region name). |
| `SUBCORTEX_LABEL_COLUMNS` | `None` | Use all columns from the Tian label file. |

In [ ]:
# ── Project root (adjust if running the notebook from a different working directory) ──
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

# ── Atlas base directories ────────────────────────────────────────────────────────────
ATLASES_DIR        = os.path.join(PROJECT_ROOT, "atlases", "deterministic")
SCHAEFER_DIR       = os.path.join(ATLASES_DIR, "schaefer_2018")
TIAN_DIR           = os.path.join(ATLASES_DIR, "Tian2020MSA_v1.4", "Tian2020MSA", "3T", "Subcortex-Only")
OUTPUT_DIR         = os.path.join(ATLASES_DIR, "self_integrated_gpt5_gemini_version")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Input NIfTI files ─────────────────────────────────────────────────────────────────
CORTICAL_NIFTI     = os.path.join(SCHAEFER_DIR, "Schaefer2018_1000Parcels_7Networks_order_FSLMNI152_1mm.nii.gz")
SUBCORTICAL_NIFTI  = os.path.join(TIAN_DIR,     "Tian_Subcortex_S4_3T_1mm.nii.gz")

# ── Optional label files ──────────────────────────────────────────────────────────────
CORTICAL_LABELS    = os.path.join(SCHAEFER_DIR, "Schaefer2018_1000Parcels_7Networks_order.txt")
SUBCORTICAL_LABELS = os.path.join(TIAN_DIR,     "Tian_Subcortex_S4_3T_label.txt")

# Column selectors for the label text files (1-based).
# Schaefer order file: col 1 = integer label, col 2 = region name.
# Tian label file: plain list of names (no explicit integer ID), col 1 = name.
CORTEX_LABEL_COLUMNS    = "2"   # region name is in column 2 of the Schaefer file
SUBCORTEX_LABEL_COLUMNS = None  # Tian file is a bare name list; use all columns

# ── Output files ──────────────────────────────────────────────────────────────────────
OUT_NIFTI      = os.path.join(OUTPUT_DIR, "Schaefer1000_TianS4_combined.nii.gz")
OUT_LABELS_CSV = os.path.join(OUTPUT_DIR, "Schaefer1000_TianS4_combined.csv")

# ── Processing options ────────────────────────────────────────────────────────────────
CORTEX_PRECEDENCE = True   # cortical labels win where both atlases are non-zero
RESAMPLE          = True   # set True if atlases have different grids; False if already aligned

# ── Verify input files exist before proceeding ────────────────────────────────────────
for label, path in [
    ("Cortical NIfTI",    CORTICAL_NIFTI),
    ("Subcortical NIfTI", SUBCORTICAL_NIFTI),
    ("Cortical labels",   CORTICAL_LABELS),
    ("Subcortical labels",SUBCORTICAL_LABELS),
]:
    status = "✓ found" if os.path.exists(path) else "✗ NOT FOUND"
    print(f"  {status}  [{label}]  {path}")

print(f"\nOutputs will be written to: {OUTPUT_DIR}")

## 3 — Helper: `load_nifti`

`load_nifti(path)` wraps `nibabel.load` with one important extra step: after
reading the image as `float32` it applies `numpy.rint` before casting to
`int64`. This corrects tiny floating-point artefacts that nibabel can introduce
during resampling — for example a voxel whose true label is `5` may be stored
as `4.999999` after interpolation, and a bare `int()` cast would silently
round it down to `4`. `numpy.rint` rounds to the nearest integer first.

**Returns** a 3-tuple: `(data: ndarray int64, affine: ndarray (4,4), header: Nifti1Header)`

In [ ]:
def load_nifti(path: str) -> Tuple[np.ndarray, np.ndarray, nib.Nifti1Header]:
    """Load a NIfTI file and return its integer data, affine, and header.

    Rounds float32 → int64 to correct floating-point resampling artefacts.
    """
    img = nib.load(path)
    data = img.get_fdata(dtype=np.float32)
    return np.rint(data).astype(np.int64), img.affine, img.header

## 4 — Load and Inspect the Cortical Atlas

We load the cortical atlas (Schaefer 1000-parcel, 7-Networks, MNI152 1 mm) and
print a summary of its shape, data type, voxel dimensions, and label range.

Expected:
- Shape: `(182, 218, 182)` for the MNI152 1 mm template.
- Non-zero labels: integers 1 – 1000 (one per cortical parcel).

In [ ]:
cort_img = nib.load(CORTICAL_NIFTI)
cort_data, cort_affine, cort_hdr = load_nifti(CORTICAL_NIFTI)

cort_labels_all  = np.unique(cort_data)
cort_nonzero     = cort_labels_all[cort_labels_all != 0]

print("─── Cortical Atlas ───────────────────────────────────")
print(f"  File       : {os.path.basename(CORTICAL_NIFTI)}")
print(f"  Shape      : {cort_data.shape}")
print(f"  Dtype      : {cort_data.dtype}")
print(f"  Voxel dims : {np.round(np.abs(np.diag(cort_affine)[:3]), 4)} mm")
print(f"  Non-zero labels : {len(cort_nonzero)}")
print(f"  Label range     : {cort_nonzero.min()} – {cort_nonzero.max()}")
print(f"  Non-zero voxels : {np.count_nonzero(cort_data):,}")

## 5 — Load and Inspect the Subcortical Atlas

We load the subcortical atlas (Tian 2020, Scale IV, 3 T, 1 mm) and print the
same summary statistics.

Expected:
- Non-zero labels: integers 1 – 54 (Scale IV has 54 subcortical parcels).
- Shape: typically `(182, 218, 182)` for the 1 mm MNI152 template.

> If the shape or affine differs from the cortical atlas, the merge step will
> raise a `ValueError` unless `RESAMPLE = True` is set in the config cell above.

In [ ]:
sub_img = nib.load(SUBCORTICAL_NIFTI)
sub_data_raw, sub_affine_raw, sub_hdr_raw = load_nifti(SUBCORTICAL_NIFTI)

sub_labels_all = np.unique(sub_data_raw)
sub_nonzero    = sub_labels_all[sub_labels_all != 0]

print("─── Subcortical Atlas ────────────────────────────────")
print(f"  File       : {os.path.basename(SUBCORTICAL_NIFTI)}")
print(f"  Shape      : {sub_data_raw.shape}")
print(f"  Dtype      : {sub_data_raw.dtype}")
print(f"  Voxel dims : {np.round(np.abs(np.diag(sub_affine_raw)[:3]), 4)} mm")
print(f"  Non-zero labels : {len(sub_nonzero)}")
print(f"  Label range     : {sub_nonzero.min()} – {sub_nonzero.max()}")
print(f"  Non-zero voxels : {np.count_nonzero(sub_data_raw):,}")

# Check grid compatibility
shape_match  = cort_data.shape == sub_data_raw.shape
affine_match = np.allclose(cort_affine, sub_affine_raw)
print()
print(f"  Shape match with cortical  : {shape_match}")
print(f"  Affine match with cortical : {affine_match}")
if not (shape_match and affine_match):
    print("  ⚠  Grids differ — resampling will be applied (RESAMPLE=True required).")

## 6 — Helper: `_parse_columns` and `_load_label_names`

### `_parse_columns(columns)`

Converts a human-friendly 1-based column selection string into a list of
0-based column indices for use with Python lists:

| Input | Output |
|-------|--------|
| `'2'` | `[1]` |
| `'2,4'` | `[1, 3]` |
| `'2-4'` | `[1, 2, 3]` |
| `'2,4-6'` | `[1, 3, 4, 5]` |
| `None` | `None` (use all columns) |

### `_load_label_names(path, columns)`

Loads a label-integer → name-string mapping from either:

- **JSON** — flat dict `{"1": "LH_Vis_1", "2": ...}`.
- **Delimited text** (TSV / CSV / TXT / whitespace-separated):
  - If the first token on a line is an integer → that integer is the label ID.
  - If the first token is *not* an integer (bare name) → the 1-based line
    number is used as the ID. This is the format used by Schaefer and Tian
    plain-text label files.
  - The `columns` parameter selects which columns to join as the name.
- Returns `{}` immediately if `path` is `None`.
- Raises `FileNotFoundError` if `path` is set but does not exist on disk.

In [ ]:
def _parse_columns(columns: Optional[str]) -> Optional[list]:
    """Convert a 1-based column selection string to a list of 0-based indices."""
    if not columns:
        return None
    cols = []
    for part in str(columns).split(','):
        part = part.strip()
        if not part:
            continue
        if '-' in part:
            a, b = part.split('-', 1)
            try:
                a_i, b_i = int(a) - 1, int(b) - 1
            except ValueError:
                continue
            cols.extend(range(a_i, b_i + 1))
        else:
            try:
                cols.append(int(part) - 1)
            except ValueError:
                continue
    return cols if cols else None


def _load_label_names(path: Optional[str], columns: Optional[str] = None) -> Dict[int, str]:
    """Load a label-to-name mapping from JSON or a delimited text file."""
    if not path:
        return {}
    if not os.path.exists(path):
        raise FileNotFoundError(f"Label file not found: {path}")

    suffix = os.path.splitext(path)[1].lower()
    if suffix == ".json":
        with open(path, "r", encoding="utf-8") as fh:
            data = json.load(fh)
        return {int(k): str(v) for k, v in data.items()}

    mapping: Dict[int, str] = {}
    cols = _parse_columns(columns)
    with open(path, "r", encoding="utf-8") as fh:
        line_idx = 0
        for line in fh:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            line_idx += 1
            if "\t" in line:
                fields = line.split("\t")
            elif "," in line:
                fields = line.split(",")
            else:
                fields = line.split()
            if len(fields) < 1:
                continue
            is_explicit_id = False
            try:
                lab = int(fields[0].strip())
                is_explicit_id = True
            except ValueError:
                lab = line_idx
            if is_explicit_id:
                name_parts = fields[1:] if cols is None else [
                    fields[ci].strip() for ci in cols if 0 <= ci < len(fields) and ci != 0]
            else:
                name_parts = fields if cols is None else [
                    fields[ci].strip() for ci in cols if 0 <= ci < len(fields)]
            name = " | ".join(p for p in name_parts if p)
            if not name and not is_explicit_id:
                name = line
            mapping[lab] = name
    return mapping


# ── Quick smoke-test: load and preview both label files ───────────────────────────────
cort_label_names = _load_label_names(CORTICAL_LABELS, columns=CORTEX_LABEL_COLUMNS)
sub_label_names  = _load_label_names(SUBCORTICAL_LABELS, columns=SUBCORTEX_LABEL_COLUMNS)

print(f"Cortical label names loaded  : {len(cort_label_names)} entries")
print(f"Subcortical label names loaded: {len(sub_label_names)} entries")
print()
print("First 5 cortical  :", dict(list(cort_label_names.items())[:5]))
print("First 5 subcortical:", dict(list(sub_label_names.items())[:5]))

## 7 — Core Integration: `integrate_atlases`

This is the heart of the pipeline. The function carries out five sub-steps:

### 7a — Optional resampling

If the two atlases don't share the same shape and affine, `integrate_atlases`
raises a `ValueError` unless `resample=True`. With resampling enabled it uses
`nibabel.processing.resample_from_to` (preferred) or
`nilearn.image.resample_to_img` (fallback), both with nearest-neighbour
interpolation (`order=0`) to preserve discrete label integers.

### 7b — Label collision avoidance

The precedent atlas's labels are kept as-is. Every non-zero label in the
non-precedent atlas is offset by the maximum label of the precedent atlas:

```
offset = max(cortical_labels)            # e.g. 1000 for Schaefer-1000
new_subcortical_label = orig + offset    # e.g. label 1 → 1001
```

This guarantees no integer can appear in both atlases after remapping.

### 7c — Array merge

```
combined = zeros_like(cort_data)
combined[cortical_nonzero_mask]               = remapped_cort  # cortical wins
combined[subcortical_nonzero & ~cortical_mask] = remapped_sub   # sub fills gaps
```

Zero (background) is never remapped and is always preserved.

### 7d — Label-name mapping

Calls `_load_label_names` for both atlases and builds the return dict:

```python
{new_label: ("cortical",    original_label, original_name),
 new_label: ("subcortical", original_label, original_name), ...}
```

### 7e — Save outputs

- Saves the combined array as `int32` NIfTI (borrowing cortical affine + header).
- Writes the CSV label table if `out_labels_csv` is provided.

In [ ]:
def save_nifti(data: np.ndarray, affine: np.ndarray, out_path: str, header=None) -> None:
    """Save a numpy array as a NIfTI file (cast to int32)."""
    nii = nib.Nifti1Image(data.astype(np.int32), affine, header=header)
    nib.save(nii, out_path)


def integrate_atlases(
    cortical_path: str,
    subcortical_path: str,
    out_nifti: str,
    out_labels_csv: Optional[str] = None,
    cortex_precedence: bool = True,
    resample: bool = False,
    cortex_labels_path: Optional[str] = None,
    subcortex_labels_path: Optional[str] = None,
    cortex_label_columns: Optional[str] = None,
    subcortex_label_columns: Optional[str] = None,
) -> Dict[int, Tuple[str, int, Optional[str]]]:
    """Merge cortical and subcortical integer-labelled NIfTI atlases into one.

    Returns
    -------
    dict mapping new_label -> (source, original_label, original_name)
    """
    cort_img = nib.load(cortical_path)
    sub_img  = nib.load(subcortical_path)

    # ── 7a: Optional resampling ────────────────────────────────────────────────
    if (cort_img.shape != sub_img.shape) or (not np.allclose(cort_img.affine, sub_img.affine)):
        if not resample:
            raise ValueError(
                f"Shape/affine mismatch: cortical {cort_img.shape} vs subcortical "
                f"{sub_img.shape}. Set resample=True to auto-resample."
            )
        logging.info("Resampling subcortical atlas to cortical grid (nearest-neighbour)…")
        if _nib_resample is not None:
            sub_img = _nib_resample(sub_img, cort_img, order=0)
        elif _nilearn_resample is not None:
            sub_img = _nilearn_resample(sub_img, cort_img, interpolation='nearest')
        else:
            raise RuntimeError(
                "No resampling backend available. "
                "Install nibabel>=3 or nilearn to enable resampling."
            )

    # ── Load arrays ────────────────────────────────────────────────────────────
    cort_data   = np.rint(cort_img.get_fdata(dtype=np.float32)).astype(np.int64)
    sub_data    = np.rint(sub_img.get_fdata(dtype=np.float32)).astype(np.int64)
    cort_affine = cort_img.affine
    cort_hdr    = cort_img.header

    cort_nonzero = np.unique(cort_data[cort_data != 0])
    sub_nonzero  = np.unique(sub_data[sub_data != 0])

    logging.info("Cortical non-zero labels  : %d  (range %d – %d)",
                 len(cort_nonzero), cort_nonzero.min(), cort_nonzero.max())
    logging.info("Subcortical non-zero labels: %d  (range %d – %d)",
                 len(sub_nonzero), sub_nonzero.min(), sub_nonzero.max())

    # ── 7b: Collision-free label remapping ────────────────────────────────────
    if cortex_precedence:
        max_cort   = int(cort_nonzero.max()) if cort_nonzero.size else 0
        offset     = max_cort
        cort_map   = {int(l): int(l)          for l in cort_nonzero}
        sub_map    = {int(l): int(l) + offset  for l in sub_nonzero}
    else:
        max_sub    = int(sub_nonzero.max()) if sub_nonzero.size else 0
        offset     = max_sub
        sub_map    = {int(l): int(l)           for l in sub_nonzero}
        cort_map   = {int(l): int(l) + offset  for l in cort_nonzero}

    logging.info("Label offset applied to non-precedent atlas: %d", offset)

    # ── 7c: Build remapped arrays and merge ───────────────────────────────────
    remapped_cort = np.zeros_like(cort_data)
    remapped_sub  = np.zeros_like(sub_data)
    for orig, new in cort_map.items():
        remapped_cort[cort_data == orig] = new
    for orig, new in sub_map.items():
        remapped_sub[sub_data == orig] = new

    combined = np.zeros_like(cort_data)
    if cortex_precedence:
        mask_cort = remapped_cort != 0
        combined[mask_cort] = remapped_cort[mask_cort]
        mask_sub_only = (remapped_sub != 0) & (~mask_cort)
        combined[mask_sub_only] = remapped_sub[mask_sub_only]
    else:
        mask_sub = remapped_sub != 0
        combined[mask_sub] = remapped_sub[mask_sub]
        mask_cort_only = (remapped_cort != 0) & (~mask_sub)
        combined[mask_cort_only] = remapped_cort[mask_cort_only]

    logging.info("Combined non-zero voxels: %d", int(np.count_nonzero(combined)))

    # ── 7d: Build return mapping with optional human-readable names ───────────
    cx_names  = _load_label_names(cortex_labels_path,    columns=cortex_label_columns)
    sub_names = _load_label_names(subcortex_labels_path, columns=subcortex_label_columns)

    mapping: Dict[int, Tuple[str, int, Optional[str]]] = {}
    for orig, new in cort_map.items():
        mapping[new] = ("cortical", orig, cx_names.get(orig))
    for orig, new in sub_map.items():
        if new in mapping:
            logging.warning("Label collision for new label %d (orig sub %d) — skipping.", new, orig)
            continue
        mapping[new] = ("subcortical", orig, sub_names.get(orig))

    # ── 7e: Save NIfTI + CSV ──────────────────────────────────────────────────
    save_nifti(combined, cort_affine, out_nifti, header=cort_hdr)
    logging.info("Saved combined atlas → %s", out_nifti)

    if out_labels_csv:
        with open(out_labels_csv, "w", newline="", encoding="utf-8") as fh:
            writer = csv.writer(fh)
            writer.writerow(["new_label", "source", "original_label", "original_name"])
            for new_label in sorted(mapping.keys()):
                src, orig, name = mapping[new_label]
                writer.writerow([new_label, src, orig, name or ""])
        logging.info("Saved label CSV → %s", out_labels_csv)

    return mapping

## 8 — Run the Integration

The single call to `integrate_atlases` below executes all seven sub-steps
described above and returns the `mapping` dict. Progress is printed via
`logging.INFO`.

In [ ]:
mapping = integrate_atlases(
    cortical_path         = CORTICAL_NIFTI,
    subcortical_path      = SUBCORTICAL_NIFTI,
    out_nifti             = OUT_NIFTI,
    out_labels_csv        = OUT_LABELS_CSV,
    cortex_precedence     = CORTEX_PRECEDENCE,
    resample              = RESAMPLE,
    cortex_labels_path    = CORTICAL_LABELS,
    subcortex_labels_path = SUBCORTICAL_LABELS,
    cortex_label_columns  = CORTEX_LABEL_COLUMNS,
    subcortex_label_columns = SUBCORTEX_LABEL_COLUMNS,
)

print(f"\nIntegration complete. Total labels in combined atlas: {len(mapping)}")

## 9 — Inspect the Return Mapping

The `mapping` dict returned by `integrate_atlases` has the form:

```python
{
  new_label: (source,        original_label, original_name),
  1:         ("cortical",    1,              "7Networks_LH_Vis_1"),
  1001:      ("subcortical", 1,              "THALAMUS-lh"),
  ...
}
```

We display the first few rows from each source to verify the remapping was
applied correctly.

In [ ]:
import pandas as pd

rows = [
    {"new_label": k, "source": v[0], "original_label": v[1], "original_name": v[2] or ""}
    for k, v in sorted(mapping.items())
]
df_mapping = pd.DataFrame(rows)

cortical_rows    = df_mapping[df_mapping["source"] == "cortical"]
subcortical_rows = df_mapping[df_mapping["source"] == "subcortical"]

print(f"Total labels  : {len(df_mapping)}")
print(f"  Cortical    : {len(cortical_rows)}")
print(f"  Subcortical : {len(subcortical_rows)}")

print("\n── First 10 cortical labels ──")
display(cortical_rows.head(10).reset_index(drop=True))

print("\n── First 10 subcortical labels ──")
display(subcortical_rows.head(10).reset_index(drop=True))

## 10 — Validate the Combined Atlas

Before treating the output as correct we perform a few sanity checks:

1. **Label count** — combined atlas should have exactly `n_cortical + n_subcortical` unique non-zero labels.
2. **No collisions** — no integer appears in both the cortical and subcortical label sets.
3. **No label loss** — every cortical and subcortical label maps to a voxel in the combined image.
4. **Voxel consistency** — total non-zero voxels in combined ≥ voxels from either individual atlas (after resampling).

In [ ]:
# Load the saved combined atlas from disk to verify the file was written correctly.
combined_img  = nib.load(OUT_NIFTI)
combined_data = np.rint(combined_img.get_fdata(dtype=np.float32)).astype(np.int64)
combined_nonzero = np.unique(combined_data[combined_data != 0])

cort_new_labels = set(k for k, v in mapping.items() if v[0] == "cortical")
sub_new_labels  = set(k for k, v in mapping.items() if v[0] == "subcortical")

# 1. Label count
expected_total = len(cort_new_labels) + len(sub_new_labels)
actual_total   = len(combined_nonzero)
label_count_ok = actual_total == expected_total

# 2. No collisions
collision = cort_new_labels & sub_new_labels
no_collision_ok = len(collision) == 0

# 3. No label loss (all mapping keys present in combined array)
present_in_combined = set(combined_nonzero.tolist())
missing_from_combined = set(mapping.keys()) - present_in_combined
no_loss_ok = len(missing_from_combined) == 0

# 4. Voxel consistency
voxels_combined = int(np.count_nonzero(combined_data))
voxels_cort_orig = int(np.count_nonzero(cort_data))

print("─── Validation Results ──────────────────────────────────")
print(f"  1. Label count match          : {'✓' if label_count_ok  else '✗'}  "
      f"(expected {expected_total}, found {actual_total})")
print(f"  2. No label collisions        : {'✓' if no_collision_ok else '✗'}  "
      f"({len(collision)} collisions)")
print(f"  3. No label loss              : {'✓' if no_loss_ok      else '✗'}  "
      f"({len(missing_from_combined)} labels missing from combined array)")
print(f"  4. Voxel count ≥ cortical     : {'✓' if voxels_combined >= voxels_cort_orig else '✗'}  "
      f"(combined={voxels_combined:,}, cortical={voxels_cort_orig:,})")
print()
print(f"  Combined label range: {combined_nonzero.min()} – {combined_nonzero.max()}")
print(f"  Cortical  label range (remapped): {min(cort_new_labels)} – {max(cort_new_labels)}")
print(f"  Subcortical label range (remapped): {min(sub_new_labels)} – {max(sub_new_labels)}")

## 11 — Inspect the CSV Label Table

The CSV written to `OUT_LABELS_CSV` provides the full provenance of every
label in the combined atlas. We load it here and display summary statistics
and a preview.

In [ ]:
df_csv = pd.read_csv(OUT_LABELS_CSV)

print(f"CSV rows  : {len(df_csv)}")
print(f"Columns   : {list(df_csv.columns)}")
print()
print("── Source counts ──")
print(df_csv["source"].value_counts().to_string())
print()
print("── Named labels (original_name not empty) ──")
named = df_csv[df_csv["original_name"].notna() & (df_csv["original_name"] != "")]
print(f"  {len(named)} / {len(df_csv)} labels have a human-readable name")
print()
print("── Last 10 rows of CSV (subcortical labels) ──")
display(df_csv.tail(10).reset_index(drop=True))

## 12 — Output File Summary

Final check: list all files written to the output directory, their sizes, and
confirm both the NIfTI and CSV are present.

In [ ]:
print(f"Output directory: {OUTPUT_DIR}\n")

files = sorted(os.listdir(OUTPUT_DIR))
for fname in files:
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    tag = ""
    if fname.endswith(".nii.gz"):
        tag = "← combined atlas NIfTI"
    elif fname.endswith(".csv"):
        tag = "← label mapping table"
    print(f"  {fname:<50}  {size_kb:>10.1f} KB  {tag}")

# Final confirmation
nifti_ok = os.path.exists(OUT_NIFTI)
csv_ok   = os.path.exists(OUT_LABELS_CSV)
print()
print(f"  Combined NIfTI written : {'✓' if nifti_ok else '✗'}  {OUT_NIFTI}")
print(f"  Label CSV written      : {'✓' if csv_ok   else '✗'}  {OUT_LABELS_CSV}")